In [1]:
# Setting Up the Environment

import sys
import os
import time
import logging
import torch
import numpy as np
from PIL import Image
import rembg
import pymeshlab as pymesh
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground
from tkinter import Tk
from tkinter.filedialog import askopenfilename
from IPython.display import Video


In [2]:
# Configure Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")


In [3]:
# Timer Utility Class
class Timer:
    def __init__(self):
        self.items = {}
        self.time_scale = 1000.0  # ms
        self.time_unit = "ms"
    
    def start(self, name: str) -> None:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.items[name] = time.time()
        logging.info(f"{name} starting...")

    def end(self, name: str) -> float:
        if name not in self.items:
            return
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = self.items.pop(name)
        delta = time.time() - start_time
        t = delta * self.time_scale
        logging.info(f"{name} finished in {t:.2f}{self.time_unit}.")
        return t

timer = Timer()


In [4]:
# Model and Parameters Setup
device = "cuda:0" if torch.cuda.is_available() else "cpu"
pretrained_model_name_or_path = "stabilityai/TripoSR"
chunk_size = 8192
foreground_ratio = 0.85
output_dir = "output"
model_save_format = "obj"
render = True
os.makedirs(output_dir, exist_ok=True)


In [5]:
# Initialize TripoSR Model
timer.start("Initializing model")
model = TSR.from_pretrained(
    pretrained_model_name_or_path,
    config_name="config.yaml",
    weight_name="model.ckpt",
)
model.renderer.set_chunk_size(chunk_size)
model.to(device)
timer.end("Initializing model")


2025-01-03 11:19:11,780 [INFO] Initializing model starting...
2025-01-03 11:19:23,206 [INFO] Initializing model finished in 11426.26ms.


11426.26166343689

In [6]:
# Load Image
file_path = "examples/stripes.png"  # Change to your file path
original_image = Image.open(file_path).convert("RGBA")
original_image = original_image.resize((512, 512))


In [8]:
from pathlib import Path
# Create a new directory for this image's output
base_name = Path(file_path).stem  # Get the base name of the file
image_dir = os.path.join(output_dir, base_name)
os.makedirs(image_dir, exist_ok=True)


In [9]:
# Save the resized original image in the new directory
original_image.save(os.path.join(image_dir, "input_resized.png"))


In [10]:
# Process Image
timer.start("Processing image")
rembg_session = rembg.new_session()
image = remove_background(original_image, rembg_session)

2025-01-03 11:20:51,755 [INFO] Processing image starting...


In [11]:
# Resize Foreground
image = resize_foreground(image, foreground_ratio)

In [12]:
# Ensure Image Mode is Correct
if image.mode != "RGBA":
    image = image.convert("RGBA")

image = np.array(image).astype(np.float32) / 255.0
image = image[:, :, :3] * image[:, :, 3:4] + (1 - image[:, :, 3:4]) * 0.5
image = Image.fromarray((image * 255.0).astype(np.uint8))


In [13]:
# Save Processed Image
image.save(os.path.join(image_dir, "input_processed.png"))
timer.end("Processing image")

2025-01-03 11:21:00,494 [INFO] Processing image finished in 8739.31ms.


8739.306211471558

In [14]:
import imageio
print(imageio.plugins)


<module 'imageio.plugins' from 'c:\\Users\\akidu\\anaconda3\\envs\\newenv\\lib\\site-packages\\imageio\\plugins\\__init__.py'>


In [15]:
import imageio
import numpy as np

# Generate test frames
test_frames = [np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8) for _ in range(30)]

# Write video using FFmpeg plugin
with imageio.get_writer("test.mp4", fps=30, codec="libx264") as writer:
    for frame in test_frames:
        writer.append_data(frame)

print("Video created successfully.")


Video created successfully.


In [16]:
# Generate 3D Model and Render
timer.start("Running model")
with torch.no_grad():
    scene_codes = model([image], device=device)
timer.end("Running model")

if render:
    timer.start("Rendering")
    render_images = model.render(scene_codes, n_views=30, return_type="pil")

    # Save render images
    for ri, render_image in enumerate(render_images[0]):
        render_image.save(os.path.join(image_dir, f"render_{ri:03d}.png"))

    # Save video of renders
    def save_video(frames, output_path, fps):
        import imageio
        import numpy as np
        with imageio.get_writer(output_path, fps=fps, codec="libx264") as writer:
            for frame in frames:
                # Convert PIL.Image to numpy array
                frame_array = np.array(frame)
                writer.append_data(frame_array)

    save_video(render_images[0], os.path.join(image_dir, "render.mp4"), fps=30)
    timer.end("Rendering")

# Exporting Mesh
timer.start("Exporting mesh")
meshes = model.extract_mesh(scene_codes, has_vertex_color=False)
mesh_file = os.path.join(image_dir, f"mesh.{model_save_format}")
meshes[0].export(mesh_file)
timer.end("Exporting mesh")

logging.info("Processing complete.")


2025-01-03 11:21:19,688 [INFO] Running model starting...
2025-01-03 11:21:44,160 [INFO] Running model finished in 24472.58ms.
2025-01-03 11:21:44,161 [INFO] Rendering starting...
2025-01-03 11:40:17,264 [INFO] Rendering finished in 1113102.12ms.
2025-01-03 11:40:17,265 [INFO] Exporting mesh starting...
2025-01-03 11:41:35,240 [INFO] Exporting mesh finished in 77975.77ms.
2025-01-03 11:41:35,241 [INFO] Processing complete.


In [17]:
# Display Render Video
Video(os.path.join(image_dir, "render.mp4"), embed=True)


In [18]:
# Exporting Mesh
timer.start("Exporting mesh")
meshes = model.extract_mesh(scene_codes, has_vertex_color=False)

# Save as .obj (optional, for compatibility)
mesh_file = os.path.join(image_dir, f"mesh.{model_save_format}")
meshes[0].export(mesh_file)

# Save as .glb
try:
    import trimesh
    # Load the .obj mesh using trimesh
    mesh_trimesh = trimesh.load(mesh_file)
    # Save as .glb
    glb_file = os.path.join(image_dir, "model.glb")
    mesh_trimesh.export(glb_file)
    logging.info(f"Mesh exported as .glb to {glb_file}")
except ImportError as e:
    logging.error("Trimesh library is required for .glb export. Please install it using 'pip install trimesh'.")
except Exception as e:
    logging.error(f"An error occurred while exporting .glb: {e}")

timer.end("Exporting mesh")
logging.info("Processing complete.")


2025-01-03 11:48:25,328 [INFO] Exporting mesh starting...
2025-01-03 11:49:45,618 [INFO] Mesh exported as .glb to output\stripes\model.glb
2025-01-03 11:49:45,618 [INFO] Exporting mesh finished in 80289.83ms.
2025-01-03 11:49:45,618 [INFO] Processing complete.
